# Performance Tuning & Best Practices

**Dataset:** `samples.tpch` + `samples.bakehouse`

**Difficulty:** Hard

**Topics:** `cache`/`persist`/`unpersist`, `repartition` vs `coalesce`, broadcast join, `explain()`, predicate pushdown, partition co-location, avoiding `collect()`, skew & salting, storage levels

> Understanding Spark's execution model is what separates engineers who write **correct** Spark from those who write **fast** Spark.
> Each problem here teaches a concrete pattern you'll use every day in production.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_lineitem     = spark.table("samples.tpch.lineitem")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_nation       = spark.table("samples.tpch.nation")
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")

## Problem 1 — Cache and Unpersist

Some DataFrames are reused multiple times in a pipeline. Caching avoids recomputing the same data.

1. Cache `df_lineitem` using `.cache()`.
2. Trigger materialisation with `.count()`.
3. Check `df_lineitem.is_cached` (should be `True`).
4. Unpersist it with `.unpersist()`.
5. Return a summary DataFrame showing each step.

**Expected output columns:** `step`, `description`, `value`

> **Rule:** Always `.unpersist()` after you're done — cached DataFrames consume executor memory and can cause OOM errors if left.

In [ ]:
# Problem 1 — write your solution here
# Build result_1 with spark.createDataFrame([('1-cache','Cached df_lineitem', str(df_lineitem.is_cached)), ...], ['step','description','value'])
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'step' in cols, "Missing column: step"
assert 'description' in cols, "Missing column: description"
assert 'value' in cols, "Missing column: value"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt >= 3, f"Expected at least 3 steps, got {cnt}"
assert not df_lineitem.is_cached, "df_lineitem should be unpersisted by end of solution"
print(f"Problem 1 passed ✓  ({cnt} steps documented)")

## Problem 2 — `repartition` vs `coalesce`

Understand how `repartition` and `coalesce` change the execution plan:

1. Build `df_repartitioned = df_transactions.repartition(8)`.
2. Build `df_coalesced = df_repartitioned.coalesce(2)`.
3. Call `.explain()` on each to observe the difference in the notebook output:
   - `repartition` adds an **Exchange** (full shuffle) node.
   - `coalesce` avoids a shuffle — it merges adjacent partitions.
4. Return a summary DataFrame with three rows describing the original, repartitioned, and coalesced states.

```python
result_2 = spark.createDataFrame([
    ("original",             spark.conf.get("spark.sql.shuffle.partitions")),
    ("after repartition(8)", "8"),
    ("after coalesce(2)",    "2"),
], ["operation", "partition_count"])
```

Cast `partition_count` to integer in the final result.

**Expected output columns:** `operation`, `partition_count`

> **Key difference:** `repartition` triggers a full shuffle (any number up or down). `coalesce` only merges existing partitions — no shuffle, but can only reduce, not increase.
>
> **Note:** Low-level partition inspection APIs (e.g. `getNumPartitions`) are unavailable on serverless compute. Use `.explain()` or Spark config to reason about partitions instead.

In [ ]:
# Problem 2 — write your solution here
# 1. Build df_repartitioned = df_transactions.repartition(8), call .explain()
# 2. Build df_coalesced = df_repartitioned.coalesce(2), call .explain()
# 3. Build a summary DataFrame with spark.createDataFrame (see markdown above)
# 4. Cast partition_count to integer
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'operation' in cols, "Missing column: operation"
assert 'partition_count' in cols, "Missing column: partition_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt == 3, f"Expected 3 rows (original, repartition, coalesce), got {cnt}"
values = [int(r['partition_count']) for r in result_2.collect()]
assert 8 in values, "repartition(8) step should show partition_count=8"
assert 2 in values, "coalesce(2) step should show partition_count=2"
print(f"Problem 2 passed ✓")

## Problem 3 — Broadcast Join for Small Tables

`samples.tpch.nation` has only 25 rows — a perfect candidate for a broadcast join.
Without broadcast, Spark would shuffle both sides. With broadcast, the small table is sent to every executor, eliminating the shuffle.

Join `df_customer` with `F.broadcast(df_nation)` on `c_nationkey = n_nationkey`.
Return the enriched DataFrame.

**Expected output columns:** `c_custkey`, `c_name`, `c_mktsegment`, `n_name`

> **Rule of thumb:** Broadcast tables smaller than ~10 MB. In Databricks the auto-broadcast threshold is `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). Explicit `F.broadcast()` is clearer and more predictable.

In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'n_name' in cols, "Missing column: n_name"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
nation_count = result_3.select('n_name').distinct().count()
assert nation_count == 25, f"Expected 25 distinct nations, got {nation_count}"
print(f"Problem 3 passed ✓  ({cnt} rows, {nation_count} nations)")

## Problem 4 — Reading the Execution Plan

Build a transformation pipeline on `df_transactions`:
1. Filter where `totalPrice > 50`
2. GroupBy `franchiseID`, `product`
3. Aggregate `total_revenue = sum(totalPrice)`

Call `.explain(mode='formatted')` on the result so you can inspect the plan in the notebook output.
Return the aggregated DataFrame as `result_4`.

**Expected output columns:** `franchiseID`, `product`, `total_revenue`

> **What to look for in the plan:**
> - `FileScan` — data is read from storage; look for pushed-down filters here (predicate pushdown)
> - `Exchange` (HashPartitioning) — a shuffle; each `groupBy` or `join` triggers one
> - `HashAggregate` — the aggregation step; appears twice (partial on each partition, then final after shuffle)

In [ ]:
# Problem 4 — write your solution here
# Build the pipeline, call .explain(mode='formatted'), then return the DataFrame
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'product' in cols, "Missing column: product"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
min_revenue = result_4.agg(F.min('total_revenue')).collect()[0][0]
assert min_revenue > 0, "total_revenue should be positive"
print(f"Problem 4 passed ✓  ({cnt} franchise-product combinations)")

## Problem 5 — Predicate Pushdown: Filter Early

**Anti-pattern:** joining two large DataFrames then filtering. Spark may push the filter down, but it's better to be explicit.

**Correct pattern:** filter BEFORE the join to reduce the data volume being shuffled.

Filter `df_transactions` to only `paymentMethod = 'Credit Card'` FIRST, then join with `df_franchises` on `franchiseID`.
Return the result.

**Expected output columns:** `franchiseID`, `name`, `country`, `product`, `total_revenue`, `transaction_count`

In [ ]:
# Problem 5 — write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'product' in cols, "Missing column: product"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6 — Co-partitioning Before a Join

When joining two large DataFrames repeatedly on the same key, pre-partitioning both DataFrames on the join key
means subsequent joins avoid a re-shuffle.

1. Repartition `df_transactions` on `franchiseID` with 10 partitions.
2. Repartition `df_franchises` on `franchiseID` with 10 partitions.
3. Join and aggregate total revenue + transaction count per franchise.

**Expected output columns:** `franchiseID`, `name`, `total_revenue`, `transaction_count`

In [ ]:
# Problem 6 — write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 6 passed ✓  ({cnt} franchises)")

## Problem 7 — Never `collect()` Large DataFrames

**Anti-pattern:** `df.collect()` pulls ALL data to the driver — for a billion-row table this crashes.

Demonstrate both approaches to find the maximum order date in `df_orders`:

- **Wrong way** (comment it out — just show it): `df_orders.collect()` then Python `max()`
- **Right way**: `df_orders.agg(F.max('o_orderdate')).collect()[0][0]`

Return a single-row DataFrame showing the result.

**Expected output columns:** `approach`, `max_order_date`

In [ ]:
# Problem 7 — write your solution here
# WRONG (do NOT run this on large data):
# max_date = max([r['o_orderdate'] for r in df_orders.collect()])  # pulls all rows to driver!

# RIGHT: aggregate first, then collect the single-row result
# max_date = df_orders.agg(F.max('o_orderdate')).collect()[0][0]
# result_7 = spark.createDataFrame([('Correct: agg then collect', str(max_date))], ['approach','max_order_date'])
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'approach' in cols, "Missing column: approach"
assert 'max_order_date' in cols, "Missing column: max_order_date"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt == 1, f"Expected 1 row, got {cnt}"
print(f"Problem 7 passed ✓")

## Problem 8 — Salting for Skewed Joins

Data skew happens when one key has far more rows than others — all data for that key lands on one partition, causing one slow task.

**Salting pattern:** add a random salt to the skewed key to spread the data across multiple partitions.

Simulate by joining `df_lineitem` (large) filtered to `l_shipmode = 'MAIL'` with a small lookup DataFrame
that maps ship modes to descriptions. Apply salting:
1. Add `salt = l_orderkey % 5` to `df_lineitem`.
2. Repartition by `(l_shipmode, salt)`.
3. Aggregate total revenue and line count per ship mode.

**Expected output columns:** `l_shipmode`, `total_revenue`, `line_count`

In [ ]:
# Problem 8 — write your solution here
# df_mail = df_lineitem.filter(F.col('l_shipmode') == 'MAIL') \
#     .withColumn('salt', F.col('l_orderkey') % 5) \
#     .repartition(10, F.col('l_shipmode'), F.col('salt'))
# Then aggregate by l_shipmode
# Assign result to: result_8

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'l_shipmode' in cols, "Missing column: l_shipmode"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'line_count' in cols, "Missing column: line_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt >= 1, "Got 0 rows"
modes = [r['l_shipmode'] for r in result_8.select('l_shipmode').collect()]
assert 'MAIL' in modes, "Expected MAIL shipmode in results"
print(f"Problem 8 passed ✓  ({cnt} ship modes)")